
# TP 6 — Mini-projet : MLP sur données tabulaires mixtes

**Objectif.** Aucune notion nouvelle en cours aujourd'hui (10 minutes de consignes seulement) : ce TP mobilise tout ce que vous avez appris depuis la séance 3 (MLP, `Trainer`, choix de loss, régularisation) sur un jeu de données tabulaire réaliste, avec un ingrédient supplémentaire : des **variables catégorielles**, encodées par des **embeddings** plutôt qu'en *one-hot*.

**Dataset.** Titanic (`sklearn.datasets.fetch_openml`) : on prédit la survie d'un passager (classification binaire) à partir de variables numériques (âge, tarif, nombre de proches à bord) et catégorielles (classe du billet, sexe, port d'embarquement).

Toute la plomberie (chargement, nettoyage minimal, encodage, `Dataset`/`DataLoader`) est déjà écrite ci-dessous : l'essentiel de votre effort porte sur les choix de modélisation (partie 1) et la recherche d'hyperparamètres (partie 3).

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import itertools

from training_toolbox import Trainer, EarlyStopping, ModelCheckpoint

torch.manual_seed(0)


## Chargement et préparation des données (fourni)

In [ ]:

from sklearn.datasets import fetch_openml
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split

titanic = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = titanic.frame

numeric_cols = ["age", "fare", "sibsp", "parch"]
categorical_cols = ["pclass", "sex", "embarked"]
target_col = "survived"

df = df[numeric_cols + categorical_cols + [target_col]].copy()

# Imputation très simple : médiane pour le numérique, modalité la plus fréquente pour le catégoriel
for col in numeric_cols:
    df[col] = df[col].astype(float).fillna(df[col].astype(float).median())
for col in categorical_cols:
    df[col] = df[col].astype("object")
    df[col] = df[col].fillna(df[col].mode()[0])

df[target_col] = df[target_col].astype(str).astype(int)

# Encodage des colonnes catégorielles en indices entiers (nécessaire pour des embeddings)
cat_encoders = {col: LabelEncoder().fit(df[col].astype(str)) for col in categorical_cols}
for col in categorical_cols:
    df[col] = cat_encoders[col].transform(df[col].astype(str))

cat_cardinalities = {col: len(cat_encoders[col].classes_) for col in categorical_cols}
print("Cardinalités des variables catégorielles :", cat_cardinalities)
print(df.head())

In [ ]:

X_num = df[numeric_cols].values.astype("float32")
X_cat = df[categorical_cols].values.astype("int64")
y = df[target_col].values.astype("float32")

X_num_train, X_num_val, X_cat_train, X_cat_val, y_train, y_val = train_test_split(
    X_num, X_cat, y, test_size=0.2, random_state=0, stratify=y
)

scaler = MinMaxScaler().fit(X_num_train)
X_num_train = scaler.transform(X_num_train).astype("float32")
X_num_val = scaler.transform(X_num_val).astype("float32")

print("Train :", X_num_train.shape, "- Val :", X_num_val.shape)


On définit un `Dataset` qui renvoie, pour chaque exemple, un dictionnaire avec les variables numériques, les variables catégorielles (indices, pour les embeddings) et le label. Le `Trainer` de `training_toolbox` sait déjà gérer des batches sous forme de dictionnaire (mécanisme initialement prévu pour les modèles Hugging Face, voir séance 11) : il retire la clé `"labels"` et passe le reste au modèle sous forme d'arguments nommés (`model(**inputs)`), ce qui convient parfaitement à un modèle prenant plusieurs entrées.

In [ ]:

class TitanicDataset(Dataset):
    def __init__(self, X_num, X_cat, y):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "numeric": self.X_num[idx],
            "categorical": self.X_cat[idx],
            "labels": self.y[idx],
        }


train_loader = DataLoader(TitanicDataset(X_num_train, X_cat_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TitanicDataset(X_num_val, X_cat_val, y_val), batch_size=64, shuffle=False)

n_numeric = X_num_train.shape[1]
print("Variables numériques :", n_numeric, "- Variables catégorielles :", list(cat_cardinalities.keys()))

In [ ]:

def binary_accuracy(preds, y):
    pred_labels = (preds > 0).float()
    return (pred_labels == y.view(-1, 1)).float().mean()


## Partie 1 — Un MLP avec embeddings pour les variables catégorielles

Un `nn.Embedding(num_embeddings, embedding_dim)` associe à chaque indice de catégorie (0, 1, 2, ...) un vecteur de dimension `embedding_dim`, appris pendant l'entraînement — c'est une alternative au *one-hot encoding* qui permet au modèle d'apprendre une représentation continue et éventuellement de capturer des similarités entre modalités (utile surtout quand le nombre de modalités est grand).

**Question 1.1.** Complétez `TabularMLP` :

- `self.embeddings` : une `nn.ModuleList` contenant, pour chaque colonne catégorielle (dans l'ordre de `cat_cardinalities`), une `nn.Embedding(cardinalité, embedding_dim)` ;
- `self.mlp` : un MLP (`nn.Sequential`, avec `nn.ReLU` et `nn.Dropout(dropout)` entre les couches cachées définies par `hidden_sizes`) qui prend en entrée la concaténation des variables numériques et de tous les embeddings (soit `n_numeric + len(cat_cardinalities) * embedding_dim` valeurs), et renvoie **un seul logit** (sortie de classification binaire, sans activation).

Dans `forward(self, numeric, categorical)` : calculez l'embedding de chaque colonne catégorielle (`self.embeddings[i](categorical[:, i])`), concaténez-les entre eux et avec `numeric` (`torch.cat(..., dim=1)`), puis passez le résultat dans `self.mlp`.

In [ ]:

class TabularMLP(nn.Module):
    def __init__(self, n_numeric, cat_cardinalities, embedding_dim=4,
                 hidden_sizes=(64, 32), dropout=0.2):
        super().__init__()
        cardinalities = list(cat_cardinalities.values())

        # TODO : self.embeddings = nn.ModuleList([nn.Embedding(card, embedding_dim) for card in cardinalities])

        input_size = n_numeric + len(cardinalities) * embedding_dim
        # TODO : construire self.mlp (nn.Sequential) : pour chaque taille de hidden_sizes,
        #        nn.Linear -> nn.ReLU -> nn.Dropout(dropout), puis une dernière nn.Linear vers 1 sortie
        #        (indice : itérez sur hidden_sizes en gardant trace de la taille d'entrée courante)

    def forward(self, numeric, categorical):
        # TODO : calculer les embeddings de chaque colonne catégorielle, les concaténer entre eux
        #        et avec `numeric`, puis passer le tout dans self.mlp
        pass


## Partie 2 — Entraîner un premier modèle

**Question 2.1.** Écrivez une fonction `build_and_train(embedding_dim=4, hidden_sizes=(64, 32), dropout=0.2, lr=1e-3, epochs=15, verbose=False)` qui instancie un `TabularMLP` avec ces hyperparamètres, l'entraîne avec le `Trainer` (`nn.BCEWithLogitsLoss()`, `torch.optim.Adam`, `metrics={"acc": binary_accuracy}`) sur `train_loader`/`val_loader` pendant `epochs` époques, et renvoie `(model, history)`.

In [ ]:

def build_and_train(embedding_dim=4, hidden_sizes=(64, 32), dropout=0.2, lr=1e-3,
                     epochs=15, verbose=False):
    # TODO : instancier TabularMLP(n_numeric, cat_cardinalities, embedding_dim, hidden_sizes, dropout)
    #        un optimizer Adam(lr=lr), un Trainer (nn.BCEWithLogitsLoss(), metrics={"acc": binary_accuracy})
    #        puis appeler trainer.fit(train_loader, val_loader, epochs=epochs, verbose=verbose)
    #        et renvoyer (model, history)
    pass


**Question 2.2.** Appelez `build_and_train` avec les valeurs par défaut (`verbose=True`), tracez les courbes de perte et d'accuracy (train/val), et affichez l'accuracy de validation finale.

In [ ]:

# TODO : model, history = build_and_train(verbose=True)
#        tracer history["train_loss"]/history["val_loss"] puis history["train_acc"]/history["val_acc"]
#        et afficher history["val_acc"][-1]


## Partie 3 — Recherche d'hyperparamètres

On compare quelques combinaisons d'hyperparamètres par une recherche *grid search* (vous pouvez aussi tirer les combinaisons aléatoirement, façon *random search*, si vous préférez) sur `embedding_dim`, `hidden_sizes` et `lr`.

**Question 3.1.** Complétez la boucle ci-dessous pour évaluer chaque combinaison de `param_grid` (appelez `build_and_train`, avec un nombre d'époques réduit pour que la recherche reste rapide, par exemple `epochs=10`), et stockez l'accuracy de validation finale de chaque combinaison dans `results`.

In [ ]:

param_grid = {
    "embedding_dim": [2, 8],
    "hidden_sizes": [(32,), (64, 32)],
    "lr": [1e-3, 1e-2],
}

results = []
for embedding_dim, hidden_sizes, lr in itertools.product(
    param_grid["embedding_dim"], param_grid["hidden_sizes"], param_grid["lr"]
):
    # TODO : model, history = build_and_train(embedding_dim=embedding_dim, hidden_sizes=hidden_sizes,
    #                                          lr=lr, epochs=10)
    #        val_acc = history["val_acc"][-1]
    #        results.append((embedding_dim, hidden_sizes, lr, val_acc))
    pass

results.sort(key=lambda r: r[-1], reverse=True)
for r in results:
    print(r)


**Question 3.2.** Reprenez la meilleure combinaison trouvée, et réentraînez-la seule avec davantage d'époques (par exemple 40) et les callbacks `EarlyStopping`/`ModelCheckpoint` vus en séance 5, pour obtenir un modèle final.

In [ ]:

# TODO : reprendre les meilleurs hyperparamètres (results[0]), instancier un TabularMLP,
#        un optimizer, un Trainer avec callbacks=[EarlyStopping(patience=5), ModelCheckpoint("best_titanic.pt")],
#        et entraîner sur 40 époques


**Questions.**
- Ici, les variables catégorielles ont une cardinalité très faible (2 à 3 modalités). Les embeddings apportent-ils un vrai gain par rapport à un simple *one-hot encoding* sur ce jeu de données ? Sur quel type de variable (ex. code postal, identifiant produit, mot d'un vocabulaire) l'argument en faveur des embeddings serait-il plus net ?
- La recherche d'hyperparamètres a-t-elle changé le classement des modèles par rapport à vos choix « par défaut » de la partie 2 ? Qu'auriez-vous pu explorer d'autre (dropout, weight decay, nombre de couches) ?


_Votre réponse ici._


## Bilan

Ce mini-projet clôt le bloc « MLP » du cours (séances 3 à 6) : architecture, optimisation, choix de loss, régularisation, variables catégorielles et recherche d'hyperparamètres. Les séances 7 à 9 passent aux CNN, pour des données image.